# DS4420 Project - Collaborative Filtering Model

Authors: Gavin Bond, Benjamin Kataoka, Preetish Paul

## User-User Collaborative Filtering

This notebook implements our user-user collaborative filtering model to predict how much developers trust AI generated output (AIAcc). We load the cleaned combined dataset exported from preprocessing.ipynb and build from there.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score

In [2]:
# Load the cleaned combined dataset from preprocessing.ipynb
df = pd.read_csv("data/combined_survey.csv")

print("Shape:", df.shape)
print("Year breakdown:")
print(df["year"].value_counts())

Shape: (55118, 14)
Year breakdown:
year
2024    29379
2025    25739
Name: count, dtype: int64


### Target Encoding

We ordinally encode AIAcc on a 0 to 4 scale rather than binarizing like the NN does. CF predicts a continuous score from similar users so keeping the full range gives us richer signal to work with.

In [3]:
trust_map = {
    "Highly trust": 4,
    "Somewhat trust": 3,
    "Neither trust nor distrust": 2,
    "Somewhat distrust": 1,
    "Highly distrust": 0
}

df["AIAcc_encoded"] = df["AIAcc"].map(trust_map)

print("AIAcc distribution:")
print(df["AIAcc_encoded"].value_counts().sort_index())

AIAcc distribution:
AIAcc_encoded
0     7490
1    13536
2    13582
3    19144
4     1366
Name: count, dtype: int64


In [4]:
# Check missing values per column before deciding how to handle them
cols_to_check = ["YearsCode", "WorkExp", "Age", "DevType", "EdLevel", "OrgSize",
                 "AISelect", "AISent", "AIThreat", "RemoteWork", "AIComplex", "ICorPM"]

missing = df[cols_to_check].isna().sum()
pct = (missing / len(df) * 100).round(1)

print(f"Total rows: {len(df)}")
print()
print(pd.DataFrame({"missing": missing, "pct": pct}).to_string())
print()
print(f"Rows remaining if we drop all missing: {df[cols_to_check].dropna().shape[0]}")

Total rows: 55118

            missing   pct
YearsCode      1442   2.6
WorkExp       12908  23.4
Age               0   0.0
DevType        1275   2.3
EdLevel        1086   2.0
OrgSize        6783  12.3
AISelect         23   0.0
AISent           39   0.1
AIThreat        828   1.5
RemoteWork     5527  10.0
AIComplex       266   0.5
ICorPM        16259  29.5

Rows remaining if we drop all missing: 37868


### Feature Cleaning

We convert the numeric columns to numbers and drop any row that has a missing value in any feature column. This brings us from 55,118 to around 37,868 developers but every row is complete with no imputation. We also keep all category values intact rather than collapsing rare ones into an "Other" bucket.

In [5]:
numeric_cols = ["YearsCode", "WorkExp"]
categorical_cols = [
    "Age", "DevType", "EdLevel", "OrgSize",
    "AISelect", "AISent", "AIThreat", "RemoteWork",
    "AIComplex", "ICorPM"
]

# Convert numeric columns to numbers
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with any missing value across all feature columns
df = df[numeric_cols + categorical_cols + ["AIAcc_encoded"]].dropna().reset_index(drop=True)

print("Rows after dropping missing:", len(df))
print("Remaining NAs:", df.isna().sum().sum())

Rows after dropping missing: 37840
Remaining NAs: 0


### Feature Encoding

We follow the same pattern from the CF notebook in class. StandardScaler on numeric columns and one-hot encoding on categoricals, then combine into one feature matrix.

In [6]:
# Scale numeric features (matching CF notebook pattern)
scaler = StandardScaler()
X_numeric = pd.DataFrame(
    scaler.fit_transform(df[numeric_cols]),
    columns=numeric_cols,
    index=df.index
)

# One-hot encode categorical features
X_categorical = pd.get_dummies(df[categorical_cols], drop_first=True)

# Combine into full feature matrix
X = pd.concat([X_numeric, X_categorical], axis=1).reset_index(drop=True)
y = df["AIAcc_encoded"].reset_index(drop=True)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (37840, 89)


### Train/Test Split

We use the cleaned dataset of around 37,868 developers with an 80/20 stratified split, giving us roughly 30,294 training developers and 7,574 test developers.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (30272, 89) | Test: (7568, 89)


### Cosine Similarity

We compute cosine similarity exactly as in the CF notebook. The one difference is that our matrix is rectangular (test x train) rather than square since we have separate train and test sets. Because of this we skip the fill_diagonal step since there is no diagonal, and we scale row-wise (axis=1) instead of column-wise (axis=0). In the notebook axis=0 scales each user's incoming similarities looking down a column. Here axis=1 is the equivalent operation since each test user's similarities to all training users sit in a row rather than a column.

In [8]:
# Compute cosine similarity between test and training developers
# Shape: n_test x n_train
similarity_matrix = cosine_similarity(X_test.to_numpy(), X_train.to_numpy())

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (7568, 30272)


In [9]:
# MinMax scale each row (matching CF notebook pattern, adapted for rectangular matrix)
sim_scores_scaled = pd.DataFrame(similarity_matrix).apply(
    lambda x: (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x)), axis=1
)

sim_scores_scaled.iloc[:3, :5]

,0,1,2,3,4
0,0.573076,0.137089,0.696616,0.448936,0.649472
1,0.320226,0.084366,0.457852,0.360244,0.310562
2,0.111535,0.877192,0.161455,0.471373,0.186236


### Weighted Average Prediction

For each test developer we find their top 20 most similar training developers and predict their AIAcc score as a similarity weighted average. This is the same prediction step from the class notebook where the more similar a neighbor is the more their rating influences the final prediction.

In [10]:
k = 20
y_pred = []

for i in range(len(X_test)):
    sim_scores = sim_scores_scaled.iloc[i]
    top_k_idx = sim_scores.nlargest(k).index
    weights = sim_scores[top_k_idx]
    neighbor_ratings = y_train.iloc[top_k_idx].values

    if weights.sum() == 0:
        y_pred.append(y_train.mean())
    else:
        y_pred.append(np.average(neighbor_ratings, weights=weights))

y_pred = np.array(y_pred)
print("Sample predictions:", y_pred[:10].round(2))
print("Actual values:     ", y_test[:10].values)

Sample predictions: [1.75 1.16 2.05 2.11 1.35 2.1  1.75 1.25 2.35 1.35]
Actual values:      [1 1 2 2 1 2 1 2 2 0]


### Evaluation

We evaluate using two metrics. First, MAE on the ordinal scale (0 to 4) which is the natural metric for CF since we are predicting a score. Second, we convert predictions to binary by thresholding at 1.5, which lets us compare directly against the neural network accuracy.

In [11]:
# MAE on ordinal scale
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE (ordinal 0-4): {mae:.4f}")

# Binary accuracy — distrust (0,1) vs trust (2,3,4)
y_pred_binary = (y_pred <= 1.5).astype(int)
y_test_binary = (y_test <= 1).astype(int)
acc = accuracy_score(y_test_binary, y_pred_binary)
print(f"Binary Accuracy:   {acc:.4f}")

MAE (ordinal 0-4): 0.6935
Binary Accuracy:   0.7639


### Developer Profile Analysis

To actually answer our research question of what factors influence distrust we attach the predictions back to the developer profiles and look at how predicted trust varies across seniority, education level, and experience.

In [12]:
# Attach predictions to test set profiles
test_original_idx = y_test.index
df_results = df.reset_index(drop=True).iloc[test_original_idx].reset_index(drop=True)
df_results["predicted_AIAcc"] = y_pred

# Average predicted trust by seniority
df_results["seniority"] = pd.cut(
    df_results["YearsCode"],
    bins=[0, 5, 10, 20, 100],
    labels=["0-5 yrs", "6-10 yrs", "11-20 yrs", "20+ yrs"]
)
print("Mean predicted AIAcc by seniority:")
print(df_results.groupby("seniority", observed=True)["predicted_AIAcc"].mean().sort_index())

Mean predicted AIAcc by seniority:
seniority
0-5 yrs      1.901228
6-10 yrs     1.842033
11-20 yrs    1.873250
20+ yrs      1.871216
Name: predicted_AIAcc, dtype: float64


In [13]:
# Average predicted trust by education level
print("Mean predicted AIAcc by EdLevel:")
print(df_results.groupby("EdLevel")["predicted_AIAcc"].mean().sort_values())

Mean predicted AIAcc by EdLevel:
EdLevel
Something else                                                                        1.820794
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)    1.821057
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       1.849645
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          1.872150
Associate degree (A.A., A.S., etc.)                                                   1.872304
Some college/university study without earning a degree                                1.876581
Primary/elementary school                                                             1.923526
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                        1.957419
Name: predicted_AIAcc, dtype: float64


In [14]:
# Correlation of numeric features with predicted AIAcc
print("Correlation with predicted AIAcc:")
print(df_results[["predicted_AIAcc", "YearsCode", "WorkExp"]].corr()["predicted_AIAcc"])

Correlation with predicted AIAcc:
predicted_AIAcc    1.000000
YearsCode          0.011825
WorkExp            0.017225
Name: predicted_AIAcc, dtype: float64


In [15]:
# Average predicted trust across all remaining categorical columns
remaining_cats = ["Age", "DevType", "OrgSize", "AISelect", "AISent", "AIThreat", "RemoteWork", "AIComplex", "ICorPM"]

for col in remaining_cats:
    print(f"Mean predicted AIAcc by {col}:")
    print(df_results.groupby(col)["predicted_AIAcc"].mean().sort_values().to_string())
    print()

Mean predicted AIAcc by Age:
Age
Under 18 years old    1.593013
18-24 years old       1.852459
25-34 years old       1.861710
35-44 years old       1.877164
45-54 years old       1.880542
55-64 years old       1.910231
65 years or older     2.037035
Prefer not to say     2.146892

Mean predicted AIAcc by DevType:
DevType
Database administrator                           1.602404
Student                                          1.634014
Engineer, site reliability                       1.733743
Project manager                                  1.744947
Educator                                         1.788613
Data or business analyst                         1.794011
System administrator                             1.807207
Developer, mobile                                1.834910
DevOps specialist                                1.840514
Developer, full-stack                            1.843200
Security professional                            1.843562
Other (please specify):                